## 1. Imports

We add the project root to `sys.path` so we can import `dataset` and `embedder`
(VAE, SelfAttention2D, LPIPS, PatchGANDiscriminator, VAECombinedLoss).

The losses module provides:
- **LPIPS** — perceptual loss via VGG16 feature maps (preserves fine details)
- **PatchGANDiscriminator** — 70×70 PatchGAN that classifies local patches as real/fake
- **VAECombinedLoss** — wraps MSE + KL(β) + LPIPS + adversarial into a single call

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

from dataset import RecordingDataset
from embedder import VAE, PatchGANDiscriminator, VAECombinedLoss, discriminator_loss
from tqdm import tqdm

## 2. Hyper-parameters

Device, batch size, epochs, learning rates, latent dim, and input size.

**Loss weights:**
- `BETA = 0.1` — reduced KL pressure so the latent carries more information
- `LPIPS_WEIGHT = 0.1` — perceptual distance helps preserve NPCs / monsters
- `ADV_WEIGHT = 0.01` — adversarial loss sharpens textures

Two separate optimisers: one for the VAE (generator) and one for the discriminator.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
EPOCHS = 50
LR_VAE = 3e-5
LR_D = 3e-5
LATENT_DIM = 256
IMG_SIZE = 256

BETA = 0.1
LPIPS_WEIGHT = 0.1
ADV_WEIGHT = 0.01

## 3. Dataset

`RecordingDataset` in `mode='vae'` scans `../try/` for `obs_*.npy` files
and returns individual frames as `(C, H, W)` tensors.

In [ ]:
dataset = RecordingDataset(data_dir="../try", game="doom", mode="vae")
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

## 4. Models

**VAE** — 5-layer encoder with attention on layers 3 and 4 (16×16 and 8×8).
The last encoder channel is capped at 256 to keep the flatten → fc bridge balanced.

**Discriminator** — 70×70 PatchGAN (4 conv layers + InstanceNorm + LeakyReLU).
It classifies each local image patch independently, forcing the VAE to generate
sharp local textures.

In [ ]:
vae = VAE(
    in_channels=3,
    latent_dim=LATENT_DIM,
    img_size=IMG_SIZE,
    encoder_channels=[32, 64, 128, 256, 256],
    encoder_kernels=[4, 4, 4, 4, 4],
    encoder_strides=[2, 2, 2, 2, 2],
    attention_layers=[3, 4],
    num_attention_heads=4,
    final_activation="sigmoid",
).to(DEVICE)

discriminator = PatchGANDiscriminator(in_channels=3, ndf=64).to(DEVICE)

## 5. Loss & Optimisers

`VAECombinedLoss` computes MSE + β·KL + LPIPS + adversarial in one forward pass.

We use AdamW for both the VAE and the discriminator. Separate optimisers are
needed because the discriminator trains on real vs. fake patches, while the VAE
tries to fool it.

In [ ]:
criterion = VAECombinedLoss(
    lpips_weight=LPIPS_WEIGHT,
    adv_weight=ADV_WEIGHT,
    beta=BETA,
).to(DEVICE)

opt_vae = optim.AdamW(vae.parameters(), lr=LR_VAE)
opt_d = optim.AdamW(discriminator.parameters(), lr=LR_D)

## 6. Training loop

Each iteration:
1. **Discriminator step** — real images → D(real)=1, reconstructions → D(fake)=0
2. **VAE step** — MSE + β·KL + LPIPS + fool discriminator

The discriminator gradients are detached from the VAE decoder's output
so they don't flow into the encoder during the D step.

Every epoch ends with a validation grid saved to `weights/doom/val_epoch_NNN.png`
and a model checkpoint via `save_pretrained`.

In [ ]:
for epoch in range(EPOCHS):
    vae.train()
    discriminator.train()
    total_loss = 0.0
    last_batch = None

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in pbar:
        x = batch.to(DEVICE)

        # ---------- VAE forward ----------
        recon_x, mu, logvar = vae(x)

        # ---------- Discriminator step ----------
        d_real = discriminator(x)
        d_fake = discriminator(recon_x.detach())
        loss_d, _ = discriminator_loss(d_real, d_fake, loss_type="lsgan")

        opt_d.zero_grad()
        loss_d.backward()
        opt_d.step()

        # ---------- VAE (generator) step ----------
        d_fake_for_vae = discriminator(recon_x)

        losses = criterion(recon_x, x, mu, logvar, d_fake_for_vae)

        opt_vae.zero_grad()
        losses["loss"].backward()
        opt_vae.step()

        total_loss += losses["loss"].item()
        last_batch = x

        pbar.set_postfix(
            loss=f"{losses['loss'].item():.2e}",
            mse=f"{losses['mse'].item():.2e}",
            lpips=f"{losses['lpips'].item():.2e}",
            adv=f"{losses['adv'].item():.2e}",
        )

    avg_loss = total_loss / len(dataloader.dataset)
    print(f"Epoch {epoch+1}/{EPOCHS}, Avg loss: {avg_loss:.4f}")

    vae.eval()
    with torch.no_grad():
        recon_vae = vae(last_batch)[0]
        n = min(4, len(last_batch))
        orig = last_batch[:n].cpu()
        recon_imgs = recon_vae[:n].cpu()

        fig, axes = plt.subplots(2, n, figsize=(3 * n, 6))
        for i in range(n):
            axes[0, i].imshow(np.transpose(orig[i].numpy(), (1, 2, 0)))
            axes[0, i].axis("off")
            axes[1, i].imshow(np.transpose(recon_imgs[i].numpy(), (1, 2, 0)))
            axes[1, i].axis("off")
        axes[0, 0].set_ylabel("Original")
        axes[1, 0].set_ylabel("Reconstruction")
        plt.suptitle(f"Epoch {epoch+1}/{EPOCHS}")
        plt.tight_layout()
        plt.savefig(f"../weights/doom/val_epoch_{epoch+1:03d}.png", dpi=150)
        plt.close()

    vae.save_pretrained(f"../weights/doom_{epoch}")
    print(f"Checkpoint saved to ../weights/doom_{epoch}")
    vae.train()

## 7. Inspect results

After training completes (or is interrupted), load a checkpoint and run a quick sanity check.
The saved `.png` grids under `weights/doom/` already show per-epoch progress.

In [ ]:
# Example: load a saved checkpoint and run a quick sanity check
# vae = VAE.from_pretrained("../weights/doom_49", map_location=DEVICE)
# vae.eval()